# Persiapan Dataset: Pemotongan Wajah (Face Cropping) DAiSEE

Buku catatan (notebook) ini bertujuan untuk memotong area wajah dari gambar-gambar dataset DAiSEE secara otomatis. Kita akan menggunakan model kecerdasan buatan dari Google (MediaPipe) untuk menemukan letak wajah, lalu memotongnya dengan menyisakan sedikit ruang ekstra (padding) agar wajah tidak terlihat terlalu ketat.

### Tahap 1: Persiapan Pustaka (Library) dan Model Kecerdasan Buatan

Pada tahap pertama ini, kita memanggil semua alat bantu yang akan digunakan:
- **OpenCV (`cv2`)**: Alat untuk membaca, memotong, dan menyimpan gambar.
- **Pandas (`pd`)**: Alat untuk membaca dan menulis data berbentuk tabel (file CSV).
- **OS & Glob**: Alat untuk membuat folder di komputer dan mencari file gambar.
- **MediaPipe**: Alat dari Google untuk menjalankan model kecerdasan buatan.

Setelah alat disiapkan, kita membangun "otak" pendeteksi wajah menggunakan file model berekstensi `.tflite`. Kita mengatur agar model hanya memproses hasil deteksi jika ia yakin minimal 50% (`0.5`) bahwa benda tersebut adalah wajah manusia.

In [ ]:
import cv2
import pandas as pd
import os
import glob
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

model_path = '' 

base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.FaceDetectorOptions(base_options=base_options, min_detection_confidence=0.5)
detector = vision.FaceDetector.create_from_options(options)


### Tahap 2: Pengaturan Lokasi Folder dan Data Label

Sebelum mulai memotong, kita harus memberi tahu program di mana letak gambar aslinya, di mana hasil potongannya harus disimpan, dan data label mana saja yang mau diproses. Kita membagi data menjadi tiga kelompok standar dalam *Machine Learning*: **Test, Train, dan Validation**.

In [ ]:
# tempat dataset gambar disimpan
base_path = ''
# tempat output disimpan (gambar)
main_output_folder = ''
# masukkan labelnya ada test, train, dan val
csv_paths = [
    '',
    '',
    ''
]

### Tahap 3: Proses Utama (Mencari, Mendeteksi, Memotong, dan Menyimpan)

Ini adalah jantung dari program kita. Kode di bawah ini akan melakukan hal-hal berikut secara berurutan:
1. Mengambil satu file CSV (misalnya `TrainLabels.csv`).
2. Membuat folder khusus bernama `Train` di dalam folder tujuan.
3. Membaca tabel CSV dan menelusuri setiap baris ID Video.
4. Mencari semua file foto `.jpg` yang berhubungan dengan ID Video tersebut.
5. Mengubah format warna gambar agar bisa dibaca oleh MediaPipe, lalu mendeteksi wajahnya. 
6. Jika wajah ditemukan, program akan mengambil koordinat kotaknya, lalu menambahkan ruang ekstra (18% ke samping, 25% ke atas, 12% ke bawah) agar tidak terpotong terlalu pas.
7. Memotong gambar berdasarkan koordinat baru tersebut dan menyimpannya ke komputer.
8. Mencatat informasi gambar baru tersebut, dan di akhir kelompok, menyimpannya menjadi file CSV baru (misalnya `Cropped_TrainLabels.csv`).

In [ ]:
for csv_path in csv_paths:
    filename = os.path.basename(csv_path)
    split_name = filename.replace('Labels.csv', '') 
    
    split_output_folder = os.path.join(main_output_folder, split_name)
    os.makedirs(split_output_folder, exist_ok=True)
    
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    
    cropped_data = []
    skip_count = 0
    
    print(f"{'-'*40}\nSedang memproses kelompok data: {split_name.upper()}\n{'-'*40}")

    for _, row in df.iterrows():
        base_id = str(row['ClipID']).replace('.avi', '')
        search_pattern = os.path.join(base_path, '*', '*', base_id, f"{base_id}*.jpg")
        
        for img_path in glob.glob(search_pattern):
            img_bgr = cv2.imread(img_path)
            
            if img_bgr is None: 
                continue

            h, w, _ = img_bgr.shape
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
            detection_result = detector.detect(mp_image)
            
            if not detection_result.detections:
                skip_count += 1
                continue

            bbox = detection_result.detections[0].bounding_box
            x_min, y_min = bbox.origin_x, bbox.origin_y
            box_w, box_h = bbox.width, bbox.height
            
            # Bagian ini diubah kalau mau menambah atau mengurangi bounding box
            pad_x = int(box_w * 0.18)      
            pad_top = int(box_h * 0.25)    
            pad_bottom = int(box_h * 0.12) 

            x1 = max(0, x_min - pad_x)
            y1 = max(0, y_min - pad_top)
            x2 = min(w, x_min + box_w + pad_x)
            y2 = min(h, y_min + box_h + pad_bottom)

            cropped_face = img_bgr[y1:y2, x1:x2]
            
            if cropped_face.size == 0: 
                continue

            img_name = os.path.basename(img_path)
            cropped_filepath = os.path.join(split_output_folder, f"face_{img_name}")
            cv2.imwrite(cropped_filepath, cropped_face)

            cropped_data.append({
                'Face_Image_Path': cropped_filepath,
                'Base_Video_ID': base_id,
                'Original_Image_Name': img_name,
                'Boredom': row['Boredom'],
                'Engagement': row['Engagement'],
                'Confusion': row['Confusion'],
                'Frustration': row['Frustration']
            })

    df_cropped = pd.DataFrame(cropped_data)
    csv_out_path = os.path.join(main_output_folder, f'Cropped_{split_name}Labels.csv')
    df_cropped.to_csv(csv_out_path, index=False)
    
    print(f"Kelompok {split_name} Selesai!")
    print(f"-> Berhasil memotong: {len(df_cropped)} wajah")
    print(f"-> Gagal atau Tidak ada wajah: {skip_count} gambar\n")
